%% [markdown]
# 03d — GIAS Schools Audit
File encoding: latin-1 (critical). Output: data/audit/schools_geocoded.parquet

In [1]:
# %%
import pandas as pd, numpy as np
from pathlib import Path
from pyproj import Transformer

In [2]:
ROOT = Path("/Users/souravamseekarmarti/Projects/aequitas")
DATA = ROOT / "data"

In [3]:
# %%
schools = pd.read_csv(DATA / "raw/poi/gias_schools.csv", encoding='latin-1', low_memory=False)
print(f"Total rows: {len(schools)}, Cols: {len(schools.columns)}")
assert len(schools) == 52278, f"FAIL: expected 52278, got {len(schools)}"
print(f"CHECK PASS: {len(schools)} rows")

Total rows: 52278, Cols: 135
CHECK PASS: 52278 rows


In [4]:
# %%
status_col = 'EstablishmentStatus (name)'
phase_col = 'PhaseOfEducation (name)'
print("Status breakdown:")
print(schools[status_col].value_counts())

Status breakdown:
EstablishmentStatus (name)
Open                           27183
Closed                         25046
Open, but proposed to close       28
Proposed to open                  21
Name: count, dtype: int64


In [5]:
# %%
open_schools = schools[schools[status_col] == 'Open'].copy()
assert len(open_schools) == 27183, f"FAIL: expected 27183 open, got {len(open_schools)}"
print(f"CHECK PASS: {len(open_schools)} open schools")
print("\nPhase breakdown (open):")
print(open_schools[phase_col].value_counts())

CHECK PASS: 27183 open schools

Phase breakdown (open):
PhaseOfEducation (name)
Primary                    16693
Not applicable              6394
Secondary                   3173
Nursery                      375
16 plus                      293
All-through                  166
Middle deemed secondary       85
Middle deemed primary          4
Name: count, dtype: int64


In [6]:
# %%
secondary_phases = ['Secondary', 'All-through', 'Middle deemed secondary']
secondary = open_schools[open_schools[phase_col].isin(secondary_phases)]
print(f"Secondary-equivalent schools: {len(secondary)}")
# 3173 secondary + 166 all-through + 85 middle deemed secondary = 3424
assert len(secondary) == 3424, f"FAIL: expected 3424, got {len(secondary)}"
print(f"CHECK PASS: {len(secondary)} secondary-equiv schools")

Secondary-equivalent schools: 3424
CHECK PASS: 3424 secondary-equiv schools


In [7]:
# %%
has_coords = open_schools['Easting'].notna() & (open_schools['Easting'] != 0)
coord_rate = has_coords.sum() / len(open_schools) * 100
print(f"Open schools with coordinates: {has_coords.sum()} / {len(open_schools)} ({coord_rate:.1f}%)")
assert coord_rate >= 95.0, f"FAIL: coord coverage {coord_rate:.1f}% below 95%"
print("CHECK PASS: coord coverage >= 95%")

Open schools with coordinates: 26506 / 27183 (97.5%)
CHECK PASS: coord coverage >= 95%


In [8]:
# %%
t = Transformer.from_crs("EPSG:27700", "EPSG:4326", always_xy=True)
work = open_schools[has_coords].copy()
lons, lats = t.transform(work['Easting'].values, work['Northing'].values)
work['longitude'] = lons
work['latitude'] = lats

In [9]:
# Filter out 6 Scottish schools (lat > 56.0) that appear in GIAS
scotland = work[work['latitude'] > 56.0]
print(f"Non-England schools (Scotland): {len(scotland)}")
print(scotland[['EstablishmentName','latitude']].to_string())
work = work[work['latitude'].between(49.8, 56.0)].copy()

Non-England schools (Scotland): 3
          EstablishmentName   latitude
32263  St Katharines School  56.338507
32264       Ardvreck School  56.383433
34433     Springwood School  57.197459


In [10]:
print(f"\nEngland open geocoded schools: {len(work)}")
assert work['latitude'].between(49.8, 56.0).all(), "FAIL: lat out of bounds"
assert work['longitude'].between(-6.4, 1.8).all(), "FAIL: lon out of bounds"
print("CHECK PASS: all coords within England bounds")


England open geocoded schools: 26503
CHECK PASS: all coords within England bounds


In [11]:
# %%
out_cols = ['URN', 'EstablishmentName', 'TypeOfEstablishment (name)',
            'PhaseOfEducation (name)', 'EstablishmentStatus (name)',
            'LA (name)', 'Postcode', 'Easting', 'Northing', 'latitude', 'longitude']
out = work[out_cols].copy()
out['is_secondary'] = out['PhaseOfEducation (name)'].isin(secondary_phases)
out.to_parquet(DATA / "audit/schools_geocoded.parquet", index=False)
print(f"Saved schools_geocoded.parquet: {len(out)} rows")
print(f"  Secondary-equiv: {out['is_secondary'].sum()}")

Saved schools_geocoded.parquet: 26503 rows
  Secondary-equiv: 3412


In [12]:
# %%
print("=== GIAS Schools Summary ===")
print(f"Total in file:      {len(schools)}")
print(f"Open schools:       {len(open_schools)}")
print(f"Open + geocoded (England): {len(out)}")
print(f"Secondary-equiv:    {out['is_secondary'].sum()}")
print(f"Coord coverage:     {coord_rate:.1f}%")
print("03d_schools_gias COMPLETE")

=== GIAS Schools Summary ===
Total in file:      52278
Open schools:       27183
Open + geocoded (England): 26503
Secondary-equiv:    3412
Coord coverage:     97.5%
03d_schools_gias COMPLETE
